## Silver Tabel Show

In [0]:
%run ../environment/env_variables

## Envirnoment Variables

- This notebook is made to be imported by the jobs of the Munic Pipeline
- The objective is set common variables used between the pipeline to easy the management of them
and centralize the definition in one place, avoiding typo errors along the pipeline

In [0]:
silver_data_path = f"{catalog_name}.{silver_schema}.munic_clean_data"

In [0]:
silver_data_tbl = spark.table(silver_data_path)

In [0]:


for col in silver_data_tbl.columns:
    print(col)

cod_ibge
ano_edicao
nome
uf_sigla
populacao_total
faixa_populacao
regiao
mtic011
mtic012
mtic013
mtic014
mtic015
mtic016
mtic017
mtic018
mtic04
mtic06
mtic071
mtic072
mtic073
mtic074
mtic075
mtic076
mtic077
mtic078
mtic08
mtic091
mtic092
mtic093
mtic10
mtic11
mtic12a1
mtic12a2
mtic12a3
mtic12a4
mtic12a5
mtic12a6
mtic12a7
mtic12a8
mtic12b1
mtic12b2
mtic12b3
mtic12b4
mtic12b5
mtic12b6
mtic12b7
mtic12b8
mtic12b9
mtic12b10
mtic12b11
mtic12b12
mtic12b13
mtic12b14
mtic131
mtic132
mtic133
mtic134
mtic135
mtic136
mtic137
mtic138
mtic139
mtic141
mtic142
mtic143
mtic144
mtic145
mtic146
mtic15
mtic161
mtic162
mtic163
mtic164
mtic165
mtic166
mtic167
mtic171
mtic172
mtic173
mtic174
mtic175
mtic176
mtic181
mtic182
mtic183
mtic184
mtic185
mtic186
mtic18a1
mtic18a2
mtic18a3
mtic18a4
mtic18a5
mtic18a6
mtic19
mtic201
mtic202
mtic203
mtic204
mtic205
mtic206
mtic211
mtic212
mtic213
mtic214
mtic215
mtic216
mtic217
mtic22
mtic23
mtic241
mtic242
mtic2421
mtic243
mtic2431
mtic244
mtic251
mtic2511
mtic2512
mti

## How to Transform `silver_munic_data` into a Dimensional Model (Star Schema)

### The Problem

Your silver table is a **wide flat table** — 5,570 rows (one per municipality) × 372 columns (7 metadata + 198 MTIC indicators + 162 MGOV indicators). This structure is problematic for BI because:

* **Schema instability** — every new MUNIC edition adds or renames indicators, requiring `ALTER TABLE` operations
* **Poor query ergonomics** — to compare indicators across regions you must reference dozens of column names individually
* **No dimensional separation** — municipality attributes, survey themes, and response values are all mixed in the same flat row
* **BI tools struggle** — dashboards built on 300+ boolean columns are slow and hard to author

### The Solution: Star Schema with an Unpivoted Fact Table

The key insight is that **survey indicator columns are not measures — they are attribute-value pairs**. Instead of keeping 361 indicator columns wide, you **unpivot (melt) them into rows**. This transforms the table from "one row per municipality with 361 columns" to "one row per municipality × indicator with a single value column."

```
Silver (wide):  cod_ibge | ano_edicao | nome | mtic011 | mtic012 | ... | mgov36
               1100015   | 2024       | Alta| true    | false   | ... | true

Gold (long):   cod_ibge | ano_edicao | cod_variable | resposta
               1100015   | 2024       | MTIC011      | true
               1100015   | 2024       | MTIC012      | false
               1100015   | 2024       | MGOV36       | true
```

### Proposed Star Schema

```
  ┌─────────────────────┐         ┌──────────────────────┐
  │   dim_municipio      │         │    dim_indicador       │
  │   (5,570 rows)       │         │    (358 rows)          │
  │   cod_ibge (PK)      │◄────────│ cod_variable (PK)     │
  │   nome, uf_sigla     │         │ ano_edicao             │
  │   regiao             │         │ meaning (from dict)    │
  │   populacao_total    │         │ tema (TIC / GOV)       │
  │   faixa_populacao    │         │ tipo_resposta (bool/cat)│
  │   porte (derived)    │         │ grupo_pergunta         │
  └──────────┬───────────┘         └───────────┬────────────┘
             │                                  │
             │    ┌──────────────────────┐       │
             │    │    dim_tempo           │       │
             │    │    ano_edicao (PK)     │       │
             │    │    decada             │       │
             │    └──────────┬───────────┘       │
             │               │                    │
             ▼               ▼                    ▼
              ┌────────────────────────────────────┐
              │     fact_survey_response            │
              │     (~2M rows per edition)           │
              │     cod_ibge (FK)                    │
              │     ano_edicao (FK)                  │
              │     cod_variable (FK)                │
              │     resposta (value as string)       │
              │     resposta_tipo (boolean/cat)      │
              │     nome, uf_sigla, regiao (denorm)  │
              └────────────────────────────────────┘
```

### Why This Design?

| Principle | How the star schema delivers it |
|---|---|
| **Schema stability** | New survey editions or indicators add **rows**, not columns — no schema changes needed |
| **BI-friendly** | Filter by `tema`, group by `regiao`, cross-tab any indicator — all from one fact table |
| **Dimensional integrity** | The existing dictionary (`munic_clean_dict`) becomes a proper conformed dimension (`dim_indicador`) |
| **Multi-year ready** | Different MUNIC editions (2022, 2024, ...) land in the same table, keyed by `ano_edicao` |
| **Query performance** | Long format with proper partitioning by `ano_edicao` is efficient for filtered aggregates |
| **Governance** | Each indicator has traceable metadata (meaning, theme, type) via `dim_indicador` — supports your Data Product Card |

### How to Build Each Component

#### 1. `dim_municipio` (Municipality Dimension)

**Source**: Metadata columns from `silver.munic_clean_data`

**Columns to extract**:
* `cod_ibge` (PK) — IBGE municipality code
* `ano_edicao` — survey edition year
* `nome` — municipality name
* `uf_sigla` — state abbreviation (e.g. RO, SP)
* `regiao` — macro-region (e.g. "1 - Norte")
* `populacao_total` — total population
* `faixa_populacao` — IBGE population band
* `porte_municipio` **(derived)** — classify as Pequeno I (≤20k), Pequeno II (≤50k), Médio (≤100k), Grande I (≤500k), Grande II (>500k)

**Why**: Separating municipality attributes from survey responses lets you group and filter by region, state, or population size without scanning indicator columns. The derived `porte_municipio` gives dashboards a clean categorical axis.

#### 2. `dim_indicador` (Indicator Dimension)

**Source**: `silver.munic_clean_dict` (358 rows: `cod_variable`, `ano_edicao`, `meaning`)

**Columns to derive**:
* `cod_variable` (PK) — e.g. MTIC011, MGOV04
* `ano_edicao` — edition year (meanings can change across years!)
* `meaning` — human-readable description from the dictionary
* `tema` — derived from prefix: `MTIC` → "TIC" (Technology & Communication), `MGOV` → "GOV" (Governance)
* `tipo_resposta` — derived from the column's Spark type in the silver table: `BooleanType` → "boolean", `StringType` → "categorical"
* `grupo_pergunta` — parent question code extracted via regex (e.g. MTIC12a1 → MTIC12, MTIC071 → MTIC07). Groups sub-items under their parent question for hierarchical drill-down.

**Why**: This is your **semantic layer**. Instead of embedding indicator metadata in column comments or notebook cells, it lives in a queryable dimension. Dashboards can display `meaning` as labels, filter by `tema`, and drill from `grupo_pergunta` to individual indicators. This directly supports your **Data Product Card** (Phase 4 requirement).

#### 3. `dim_tempo` (Time Dimension)

**Source**: Distinct `ano_edicao` values from the silver table.

**Columns**:
* `ano_edicao` (PK) — survey edition year
* `decada` — derived (e.g. 2024 → "2020s")

**Why**: Even with a single edition today, this dimension future-proofs the model for multi-year comparisons. It also gives BI tools a proper time axis rather than filtering on an integer column.

#### 4. `fact_survey_response` (Fact Table — Long Format)

**Source**: `silver.munic_clean_data` — all 361 indicator columns unpivoted.

**Transformation**: Use Spark's `stack()` function (or `unpivot`) to melt the 361 indicator columns into rows:

* **Input**: 5,570 rows × 372 columns
* **Output**: ~2,010,770 rows (5,570 × 361) × 8 columns

**Columns**:
* `cod_ibge` (FK → dim_municipio)
* `ano_edicao` (FK → dim_tempo)
* `cod_variable` (FK → dim_indicador)
* `resposta` — the indicator value cast to string ("true", "false", or categorical text like "Banda larga")
* `resposta_tipo` — "boolean" or "categorical" (copied from dim_indicador)
* `nome`, `uf_sigla`, `regiao`, `faixa_populacao` — denormalized for query convenience (avoid joins in common dashboard queries)

**Partitioning**: Partition by `ano_edicao` so each edition is a separate partition — efficient for incremental loads and time-scoped queries.

**Why long format?**
* Adding a new indicator in a future MUNIC edition = adding rows, not columns
* A single `WHERE cod_variable = 'MTIC014'` replaces remembering that MTIC014 is column #11
* Aggregations like "% of municipalities with WhatsApp by region" become a simple `GROUP BY regiao` with a `WHERE cod_variable = 'MTIC014' AND resposta = 'true'`
* The same fact table serves ALL dashboard questions — no need for separate fact tables per theme

#### 5. Aggregation Views (Pre-computed Gold Layer)

On top of the fact table, create views for common BI patterns:

* **`agg_indicador_regiao`** — For each boolean indicator, compute `% of municipalities where resposta = 'true'`, grouped by `regiao` and `tema`. Joined with `dim_indicador` for human-readable labels.
* **`agg_indicador_uf`** — Same, grouped by `uf_sigla` (state level).
* **`agg_indicador_porte`** — Same, grouped by `porte_municipio` (population size).

These views pre-compute the most common dashboard metric: **"What percentage of municipalities have X?"** sliced by a geographic or demographic dimension.

### How This Maps to Your Phase 3 & 4 Requirements

| Phase 3 (Development) | How the star schema delivers |
|---|---|
| Pipeline (Silver → Gold) | The unpivot + dimension extraction IS the transformation logic. Track versions of this script as your "experiment log" |
| Semantic layer / Analytical output | `dim_indicador` + `fact_survey_response` = the semantic layer that answers business questions |
| Evaluation: Data completeness | FK integrity checks (all `cod_variable` in fact → exists in `dim_indicador`; all `cod_ibge` → exists in `dim_municipio`) |
| Evaluation: Query performance | Long format with partitioning enables fast filtered aggregates; track execution time of key dashboard queries |
| Evaluation: Business alignment | `dim_indicador.meaning` + `tema` ensure metrics match the business glossary (your dictionary) |

| Phase 4 (Governance) | How the star schema delivers |
|---|---|
| Versioning | Gold Delta tables support time travel; SQL pipeline code tracked in Git |
| Metadata / Data Dictionary | `dim_indicador` IS the data dictionary, queryable and versioned as a Delta table |
| Data Product Card | Document each aggregation view: objective, source tables, metric definitions, target audience, refresh schedule |
| Monitoring | Row count anomalies (fact row count should = municipalities × indicators per edition), null checks on `resposta`, FK integrity alerts |

### Semantic Layer: A Separate Schema & Pipeline Notebook

The dimensional tables (`dim_municipio`, `dim_indicador`, `dim_tempo`, `fact_survey_response`) live in the **gold** schema and represent the raw analytical foundation. On top of them, you should create a **semantic schema** (e.g. `munic_catalog.semantic`) that holds the aggregation views, curated metrics, and business-ready datasets that dashboards consume directly.

```
munic_catalog.gold               →  Atomic dimensions + fact table (raw star schema)
  dim_municipio
  dim_indicador
  dim_tempo
  fact_survey_response

munic_catalog.semantic           →  Curated views & metric tables (dashboard-ready)
  agg_indicador_regiao
  agg_indicador_uf
  agg_indicador_porte
  kpi_resumo_municipal
  metric_transparency_score
  ...
```

**Why a separate schema?**

* **Separation of concerns** — gold holds the *how* (unpivoted atomic data), semantic holds the *what* (business metrics). Dashboard authors query semantic only and never touch the raw fact table.
* **Governed metrics** — each view in `semantic` is a governed, documented metric definition. This is where your "business glossary" lives as code. Changing a formula means updating one view, not every dashboard.
* **Access control** — you can grant BI users `SELECT` on `semantic` only, keeping gold tables restricted to pipeline processes.
* **Versioning clarity** — gold tables change when the pipeline runs (new data arrives); semantic views change when business logic changes (formula revised). Keeping them separate makes the Git history cleaner.

**Notebook: `gold_to_semantic` (or `gold_to_analytics`)**

Create a dedicated pipeline notebook at `/Shared/MUNIC_Governance/pipeline/gold_to_semantic` that:

1. **Reads** the gold fact table and dimensions (via `%run ../environment/env_variables` for catalog/schema names).
2. **Defines** each semantic view as a `CREATE OR REPLACE VIEW` SQL statement (or `CREATE OR REPLACE TABLE` for materialized metrics that are expensive to recompute).
3. **Writes** them into `munic_catalog.semantic`.
4. **Logs** the run (version, timestamp, which views were created/replaced) — this is your Phase 3 "experiment log" for the semantic layer.

```
Pipeline flow:

  landing_to_bronze  →  bronze_to_silver  →  silver_to_gold  →  gold_to_semantic
  (XLSX → Delta)     (Clean + merge)      (Unpivot + dims)  (Aggregation views + KPIs)
```

This four-stage pipeline gives you full traceability: raw data (bronze) → cleaned (silver) → dimensional model (gold) → curated analytics (semantic). Each stage is a separate notebook in Git, and each can be versioned and re-run independently.